# Projeto De Machine Learning - 2026.1

## Notebook 07 — Resumo Estruturado por Template

### Projeto Kyra Pesquisa

*Maria Beatriz Ribeiro, Juliane Oliveira e Emanuel Gandra*

## 1. Objetivo

Este notebook é o **documento de entrega final** do pipeline Kyra. Ele não re-treina modelos —
carrega os outputs dos notebooks anteriores e os consolida num relatório estruturado pronto para
apresentação e entrega.

---

### Seções

| Seção | Conteúdo | Fonte |
|---|---|---|
| **2. Setup** | Caminhos e carregamento de todos os outputs | — |
| **3. Visão Geral do Corpus** | KPIs: chunks, documentos, projetos, idioma | NB 01–02 |
| **4. Resumo por Projeto** | Temas dominantes, sentimento, termos-chave por projeto | NB 03, 04, 06 |
| **5. Tópicos Robustos** | Pares LDA × NMF confirmados por Jaccard | NB 04 |
| **6. Insights por Segmento** | Top insights com lift por projeto e público | NB 03 |
| **7. Evidências Citáveis** | Top-3 trechos representativos por tópico NMF | NB 05 |
| **8. Análise de Sentimento** | Distribuição POS/NEU/NEG por projeto e cluster | NB 06 |
| **9. Validação do Modelo** | Baseline, leakage, confusões, erros reais | NB 04 |
| **10. Export Final** | `relatorio_final.json`, `relatorio_final.md`, CSVs | — |

---

**Inputs:** outputs dos notebooks 01–06  
**Outputs:** `outputs/relatorio_final/`

---
## 2. Setup e Carregamento

In [1]:
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)

# ── Detecta raiz do projeto ──────────────────────────────────────────────
_cwd = Path(os.path.abspath(''))
_root = _cwd
for _ in range(4):
    if (_root / 'data' / 'processed').exists():
        break
    _root = _root.parent

PROCESSED_DIR  = _root / 'data' / 'processed'
CLUSTER_BASE   = _root / 'outputs' / 'clusterizacao_insights_v2'
CLUSTER_RUN    = sorted(CLUSTER_BASE.glob('*'), reverse=True)[0]
VALIDACAO_DIR  = _root / 'outputs' / 'validacao'
TABELAS_DIR    = _root / 'outputs' / 'tabelas'
XAI_DIR        = _root / 'outputs' / 'xai'
SENT_DIR       = _root / 'outputs' / 'sentimento'
RELATORIO_DIR  = _root / 'outputs' / 'relatorio_final'
RELATORIO_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raiz do projeto  : {_root}')
print(f'Run selecionada  : {CLUSTER_RUN.name}')
print(f'Output relatório : {RELATORIO_DIR}')

Raiz do projeto  : /Users/I759133/Library/CloudStorage/OneDrive-SAPSE/Documentos Juliane/IBMEC/SEXTO SEMESTRE/PROJETO/kyrapesquisa/Kyra Atual
Run selecionada  : 20260520_113457
Output relatório : /Users/I759133/Library/CloudStorage/OneDrive-SAPSE/Documentos Juliane/IBMEC/SEXTO SEMESTRE/PROJETO/kyrapesquisa/Kyra Atual/outputs/relatorio_final


In [2]:
# ── Carrega todos os outputs ──────────────────────────────────────────────

# Base principal
df_base = pd.read_parquet(PROCESSED_DIR / 'interviews_chunks_modeling.parquet')

# Clusters + tópicos
df_clusters = pd.read_parquet(CLUSTER_RUN / 'chunks_with_clusters_topics.parquet')
df_nmf      = pd.read_csv(CLUSTER_RUN / 'tables' / 'nmf_topics.csv')
df_lda      = pd.read_csv(CLUSTER_RUN / 'tables' / 'lda_topics.csv')
df_km_labels= pd.read_csv(CLUSTER_RUN / 'tables' / 'cluster_tfidf_labels.csv')
df_km_repr  = pd.read_csv(CLUSTER_RUN / 'tables' / 'cluster_representative_examples.csv')
df_ins_km   = pd.read_csv(CLUSTER_RUN / 'tables' / 'insights_clusters_by_segment.csv')
df_ins_nmf  = pd.read_csv(CLUSTER_RUN / 'tables' / 'insights_nmf_topics_by_segment.csv')

# Validação
df_class    = pd.read_csv(VALIDACAO_DIR / 'classificacao_projeto_resultados.csv')
df_jaccard  = pd.read_csv(VALIDACAO_DIR / 'topicos_robustos_jaccard.csv')
df_coh      = pd.read_csv(VALIDACAO_DIR / 'lda_coerencia_cv.csv')

# XAI
df_evidencias = pd.read_csv(XAI_DIR / 'top_trechos_nmf.csv')
df_taxonomy   = pd.read_csv(XAI_DIR / 'clusters_taxonomy_nb.csv')
with open(XAI_DIR / 'evidencias_por_topico.json', encoding='utf-8') as f:
    evidencias_json = json.load(f)

# Sentimento
df_sent_proj    = pd.read_csv(SENT_DIR / 'sentimento_lexico_por_projeto.csv')
df_sent_cluster = pd.read_csv(SENT_DIR / 'sentimento_lexico_por_cluster.csv')
df_sent_nmf_top = pd.read_csv(SENT_DIR / 'sentimento_lexico_por_topico_nmf.csv')
df_exemplos_sent= pd.read_csv(SENT_DIR / 'exemplos_sentimento_lexico_por_projeto.csv')
df_sent_lexico  = pd.read_parquet(SENT_DIR / 'chunks_sentimento_lexico.parquet')

# Tabelas XAI extra (NB 04 seção 10)
_baseline_path = TABELAS_DIR / 'comparacao_baseline_modelos.csv'
_leakage_path  = TABELAS_DIR / 'diagnostico_leakage_semantico.csv'
_erros_path    = TABELAS_DIR / 'exemplos_erros_classificacao.csv'
df_baseline = pd.read_csv(_baseline_path) if _baseline_path.exists() else pd.DataFrame()
df_leakage  = pd.read_csv(_leakage_path)  if _leakage_path.exists()  else pd.DataFrame()
df_erros    = pd.read_csv(_erros_path)    if _erros_path.exists()    else pd.DataFrame()

print('Carregamento concluído.')
print(f'  Base principal : {df_base.shape[0]:,} chunks | {df_base["projeto"].nunique()} projetos')
print(f'  Clusters       : {df_clusters["cluster_label"].nunique()} clusters | {df_nmf.shape[0]} tópicos NMF | {df_lda.shape[0]} tópicos LDA')
print(f'  Sentimento     : {df_sent_lexico["sentimento_lexico"].value_counts().to_dict()}')

Carregamento concluído.
  Base principal : 4,976 chunks | 12 projetos
  Clusters       : 18 clusters | 18 tópicos NMF | 10 tópicos LDA
  Sentimento     : {'POS': 4048, 'NEU': 593, 'NEG': 335}


---
## 3. Visão Geral do Corpus

Resumo quantitativo da base de entrevistas processada pelo pipeline.

In [3]:
# ── KPIs gerais ─────────────────────────────────────────────────────────
n_chunks    = df_base.shape[0]
n_docs      = df_base['doc_id'].nunique()
n_projetos  = df_base['projeto'].nunique()
n_clusters  = df_clusters['cluster_label'].nunique()
n_nmf       = df_nmf.shape[0]
n_lda       = df_lda.shape[0]

dist_sent = df_sent_lexico['sentimento_lexico'].value_counts(normalize=True).mul(100).round(1)

kpis = {
    'chunks_analisados':  n_chunks,
    'documentos':         n_docs,
    'projetos':           n_projetos,
    'clusters_kmeans':    n_clusters,
    'topicos_nmf':        n_nmf,
    'topicos_lda':        n_lda,
    'pct_pos_geral':      float(dist_sent.get('POS', 0)),
    'pct_neu_geral':      float(dist_sent.get('NEU', 0)),
    'pct_neg_geral':      float(dist_sent.get('NEG', 0)),
}

print('=' * 55)
print('KYRA PESQUISA — VISÃO GERAL DO CORPUS')
print('=' * 55)
for k, v in kpis.items():
    print(f'  {k:<28}: {v}')

# Distribuição de chunks por projeto
proj_dist = (
    df_base.groupby('projeto')
    .agg(n_chunks=('chunk_id','count'), n_docs=('doc_id','nunique'))
    .sort_values('n_chunks', ascending=False)
    .reset_index()
)
proj_dist['share_pct'] = (proj_dist['n_chunks'] / proj_dist['n_chunks'].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(proj_dist)))
bars = ax.barh(proj_dist['projeto'], proj_dist['n_chunks'], color=colors)
for i, (_, row) in enumerate(proj_dist.iterrows()):
    ax.text(row['n_chunks'] + 10, i, f"{row['n_chunks']:,}  ({row['share_pct']:.1f}%)",
            va='center', fontsize=9)
ax.set_xlabel('Número de chunks')
ax.set_title('Distribuição de Chunks por Projeto', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(RELATORIO_DIR / 'fig_01_chunks_por_projeto.png', dpi=150, bbox_inches='tight')
plt.show()

display(proj_dist)
proj_dist.to_csv(RELATORIO_DIR / 'corpus_por_projeto.csv', index=False)

KYRA PESQUISA — VISÃO GERAL DO CORPUS
  chunks_analisados           : 4976
  documentos                  : 131
  projetos                    : 12
  clusters_kmeans             : 18
  topicos_nmf                 : 18
  topicos_lda                 : 10
  pct_pos_geral               : 81.4
  pct_neu_geral               : 11.9
  pct_neg_geral               : 6.7


,projeto,n_chunks,n_docs,share_pct
0,natura_3cs,985,44,19.8
1,rosacea,539,10,10.8
2,mercato_brasil,516,9,10.4
3,compras_digitais,492,15,9.9
4,3cs_perfumes,485,17,9.7
5,radiosa,474,8,9.5
6,gaia_ii,416,8,8.4
7,anima,406,8,8.2
8,biome,236,4,4.7
9,jack_pearson,200,4,4.0


---
## 4. Resumo por Projeto

Para cada projeto: tema dominante (NMF), sentimento (léxico), termos-chave (Chi²) e rótulo taxonômico (NB).

In [4]:
# ── Constrói tabela resumo por projeto ───────────────────────────────────

# Tema NMF dominante por projeto (maior share)
df_nmf_proj = (
    df_clusters.groupby(['projeto', 'nmf_topic_id'])
    .size()
    .reset_index(name='n')
)
nmf_label_map = dict(zip(df_nmf['topic_id'], df_nmf['auto_label_short']))
df_nmf_proj['nmf_label'] = df_nmf_proj['nmf_topic_id'].map(nmf_label_map)
tema_dom_proj = (
    df_nmf_proj.sort_values('n', ascending=False)
    .drop_duplicates('projeto')
    .set_index('projeto')[['nmf_topic_id', 'nmf_label', 'n']]
    .rename(columns={'nmf_topic_id': 'tema_nmf_id', 'nmf_label': 'tema_nmf_label', 'n': 'n_chunks_tema'})
)

# Sentimento por projeto
sent_idx = df_sent_proj.set_index(df_sent_proj.columns[0]) if 'projeto' not in df_sent_proj.columns else df_sent_proj.set_index('projeto')
for col in ['POS', 'NEG', 'NEU']:
    if col not in sent_idx.columns:
        sent_idx[col] = 0

# Score léxico médio por projeto
score_lexico_proj = (
    df_sent_lexico.groupby('projeto')['score_lexico']
    .mean().round(2)
    .rename('score_lexico_medio')
)

# Top 5 termos Chi² (carrega se disponível)
_chi2_path = VALIDACAO_DIR / 'chi2_termos_projeto.csv'
if _chi2_path.exists():
    df_chi2 = pd.read_csv(_chi2_path)
    top_termos_proj = (
        df_chi2[df_chi2['rank'] <= 5]
        .groupby('classe')['termo']
        .apply(lambda x: ' · '.join(x))
        .rename('top5_termos_chi2')
    )
else:
    top_termos_proj = pd.Series(dtype=str, name='top5_termos_chi2')

# Monta tabela final
df_resumo_proj = proj_dist.set_index('projeto').join(tema_dom_proj, how='left')
df_resumo_proj = df_resumo_proj.join(sent_idx[['POS', 'NEU', 'NEG']].rename(
    columns={'POS':'sent_pct_pos', 'NEU':'sent_pct_neu', 'NEG':'sent_pct_neg'}), how='left')
df_resumo_proj = df_resumo_proj.join(score_lexico_proj, how='left')
df_resumo_proj = df_resumo_proj.join(top_termos_proj, how='left')
df_resumo_proj = df_resumo_proj.sort_values('n_chunks', ascending=False)

print('Tabela Resumo por Projeto:\n')
display(df_resumo_proj[['n_chunks', 'n_docs', 'tema_nmf_label', 'sent_pct_pos', 'sent_pct_neg', 'score_lexico_medio', 'top5_termos_chi2']])
df_resumo_proj.to_csv(RELATORIO_DIR / 'resumo_por_projeto.csv')

Tabela Resumo por Projeto:



,n_chunks,n_docs,tema_nmf_label,sent_pct_pos,sent_pct_neg,score_lexico_medio,top5_termos_chi2
projeto,,,,,,,
natura_3cs,985,44,presente / dar / presentear / gosta,82.4,7.0,3.31,presente · presentes · presentear · aniversario · maes
rosacea,539,10,propaganda / impressao / propagandas / mulheres,84.8,3.9,3.16,propaganda · acido · amazonia · hialuronico · acido hialuronico
mercato_brasil,516,9,pedido / fusao / lider / avon,63.8,17.2,1.62,fusao · lider · avo · avon · pedido
compras_digitais,492,15,revista / site / compra / comprar,82.5,4.7,3.03,site · interativa · revista · revista interativa · espaco digital
3cs_perfumes,485,17,loja / lojas / boticario / experiencia,74.0,7.8,2.03,perfumaria · perfumes · boticario · perfume · mari
radiosa,474,8,negra / pele negra / pele / acha,85.0,5.5,3.68,pele negra · negra · pele · nivea · corpo
gaia_ii,416,8,transmite / atencao / parece / embalagem,87.7,3.4,3.47,transmite · tecnologia avancada · avancada · transmitem · marcia
anima,406,8,refil / embalagem / parece / formato,91.6,3.7,4.79,refil · formato · atual · anc · plastico
biome,236,4,vegano / bebe / produto / produto vegano,84.3,5.5,3.16,biome · lixo · barra · lola · rendimento


In [5]:
# ── Heatmap sentimento × projeto ─────────────────────────────────────────
heat_cols = ['sent_pct_pos', 'sent_pct_neu', 'sent_pct_neg']
heat_df   = df_resumo_proj[heat_cols].copy().sort_values('sent_pct_neg', ascending=False)
heat_df.columns = ['POS %', 'NEU %', 'NEG %']

fig, ax = plt.subplots(figsize=(6, max(5, len(heat_df) * 0.55)))
sns.heatmap(
    heat_df, annot=True, fmt='.1f', cmap='RdYlGn',
    vmin=0, vmax=100, linewidths=0.5, ax=ax,
    cbar_kws={'label': '% de chunks'}
)
ax.set_title('Sentimento Léxico por Projeto', fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(RELATORIO_DIR / 'fig_02_sentimento_por_projeto.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Tópicos Robustos — LDA × NMF (Jaccard)

Tópicos confirmados por **dois modelos independentes** (LDA e NMF) via sobreposição de vocabulário.
Jaccard ≥ 0.25 indica evidência convergente — o tópico não é artefato de um único método.

In [6]:
# ── Tabela de tópicos robustos ────────────────────────────────────────────
print(f'Pares LDA × NMF robustos (Jaccard ≥ 0.25): {len(df_jaccard)} de {df_lda.shape[0]}\n')

display(
    df_jaccard[['lda_idx', 'lda_label', 'nmf_idx', 'nmf_label', 'jaccard', 'termos_comuns']]
    .sort_values('jaccard', ascending=False)
    .rename(columns={
        'lda_label': 'Tópico LDA', 'nmf_label': 'Tópico NMF',
        'jaccard': 'Jaccard', 'termos_comuns': 'Termos em Comum'
    })
)

# ── Visualização Jaccard ──────────────────────────────────────────────────
df_jac_plot = df_jaccard.sort_values('jaccard', ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if v >= 0.5 else '#f39c12' if v >= 0.35 else '#e67e22' for v in df_jac_plot['jaccard']]
bars = ax.barh(
    [f"LDA: {r['lda_label'][:30]}" for _, r in df_jac_plot.iterrows()],
    df_jac_plot['jaccard'],
    color=colors
)
ax.axvline(0.25, color='red', linestyle='--', linewidth=1, label='Limiar robusto (0.25)')
ax.axvline(0.50, color='green', linestyle='--', linewidth=1, label='Alta concordância (0.50)')
ax.set_xlabel('Jaccard (sobreposição de vocabulário)')
ax.set_title('Top 10 Pares Robustos LDA × NMF', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(RELATORIO_DIR / 'fig_03_jaccard_robustos.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabela de coerência
print('\nCoerência C_v (Gensim) por K (LDA):')
display(df_coh.sort_values('coerencia_cv', ascending=False).head(6))
best_k = df_coh.loc[df_coh['coerencia_cv'].idxmax(), 'k']
print(f'  → K com maior C_v: {best_k}')

Pares LDA × NMF robustos (Jaccard ≥ 0.25): 10 de 10



,lda_idx,Tópico LDA,nmf_idx,Tópico NMF,Jaccard,Termos em Comum
0,0,embalagem / atencao / produto / parece,8,transmite / atencao / parece / embalagem,0.714,"atencao, chama, chama atencao, embalagem, imagem, parece"
5,5,pele / corpo / uso / gosto,3,corpo / banho / uso / hidratante,0.600,"banho, cheiro, corpo, creme, gosto, hidratante"
2,2,loja / boticario / natura / experiencia,10,loja / lojas / boticario / experiencia,0.500,"boticario, experiencia, loja, lojas, natura, perfumaria"
4,4,natura / perfume / vende / cliente,0,boticario / natura / perfume / vende,0.500,"boticario, clientes, marca, marcas, natura, perfume"
6,6,presente / mae / dar / gosto,5,presente / dar / presentear / gosta,0.500,"comprar, dar, gosta, gosto, mae, maes"
1,1,natura / tempo / pedido / depois,12,pedido / fusao / lider / avon,0.333,"avon, fusao, lider, natura, pedido, teve"
8,8,comprar / revista / compra / site,4,revista / site / compra / comprar,0.333,"compra, comprar, compro, consultora, revista, site"
9,9,nome / idade / comecar / papo,13,nome / comecar / papo / vontade,0.333,"comecar, nome, ouvir, papo, prazer, vontade"
3,3,produto / embalagem / parece / ver,7,refil / embalagem / parece / formato,0.263,"diferente, embalagem, legal, parece, refil"
7,7,produto / pele / propaganda / natura,1,propaganda / impressao / propagandas / mulheres,0.263,"impressao, marca, pele, produto, propaganda"



Coerência C_v (Gensim) por K (LDA):


,k,coerencia_cv,perplexidade
1,8,0.519615,1565.992675
3,12,0.513845,1651.076147
5,18,0.502516,1687.654058
2,10,0.501689,1609.536504
0,6,0.475885,1559.151965
4,15,0.457761,1646.766100


  → K com maior C_v: 8


---
## 6. Principais Insights por Segmento

Combinações segmento–tema com **lift** mais alto — indicam onde um tema aparece com
frequência significativamente acima da média global.

In [7]:
# ── Top insights KMeans por segmento ────────────────────────────────────
LIFT_MIN    = 5.0
N_CHUNKS_MIN = 50

# Detecta colunas disponíveis
segment_col  = 'segment_col'  if 'segment_col'  in df_ins_km.columns else df_ins_km.columns[0]
segment_val  = 'segment_value' if 'segment_value' in df_ins_km.columns else df_ins_km.columns[1]
label_col    = next((c for c in df_ins_km.columns if 'label' in c.lower()), df_ins_km.columns[2])
lift_col     = next((c for c in df_ins_km.columns if 'lift' in c.lower()), None)
n_col        = next((c for c in df_ins_km.columns if 'n_chunks' in c.lower()), None)

df_top_ins = df_ins_km.copy()
if lift_col:
    df_top_ins = df_top_ins[df_top_ins[lift_col] >= LIFT_MIN]
if n_col:
    df_top_ins = df_top_ins[df_top_ins[n_col] >= N_CHUNKS_MIN]
if lift_col:
    df_top_ins = df_top_ins.sort_values(lift_col, ascending=False)

print(f'Top insights KMeans (lift ≥ {LIFT_MIN}, n_chunks ≥ {N_CHUNKS_MIN}):\n')
cols_show = [c for c in [segment_col, segment_val, label_col, lift_col, n_col] if c]
display(df_top_ins[cols_show].head(20))
df_top_ins.to_csv(RELATORIO_DIR / 'insights_top_por_segmento.csv', index=False)

Top insights KMeans (lift ≥ 5.0, n_chunks ≥ 50):



,segment_col,segment_value,label_col,lift_vs_global,n_chunks_segment_label
0,projeto,havana_iii,cluster_label,39.492063,122
1,projeto,jack_pearson,cluster_label,24.880000,170
6,publico,consumidoras,cluster_label,15.220065,190
9,projeto,bem_maes,cluster_label,13.570641,92
2,projeto,anima,cluster_label,12.021108,358
8,projeto,gaia_ii,cluster_label,11.904306,208
11,projeto,gaia_ii,cluster_label,10.409771,161
12,projeto,radiosa,cluster_label,10.365841,157
10,projeto,biome,cluster_label,9.993926,164
3,projeto,compras_digitais,cluster_label,9.623724,432


In [8]:
# ── Heatmap lift: projetos × tópicos NMF ────────────────────────────────
if 'projeto' in df_ins_nmf.get(segment_col, pd.Series()).values if segment_col in df_ins_nmf.columns else False:
    pass  # já filtrado abaixo

seg_col_nmf = 'segment_col'   if 'segment_col'   in df_ins_nmf.columns else df_ins_nmf.columns[0]
seg_val_nmf = 'segment_value' if 'segment_value' in df_ins_nmf.columns else df_ins_nmf.columns[1]
lbl_nmf     = next((c for c in df_ins_nmf.columns if 'label' in c.lower()), df_ins_nmf.columns[2])
lift_nmf    = next((c for c in df_ins_nmf.columns if 'lift' in c.lower()), None)

# Filtra apenas segmento = projeto
df_ins_proj = df_ins_nmf[df_ins_nmf[seg_col_nmf] == 'projeto'].copy()

if not df_ins_proj.empty and lift_nmf:
    pivot = df_ins_proj.pivot_table(
        index=seg_val_nmf, columns=lbl_nmf, values=lift_nmf, aggfunc='max'
    ).fillna(0)

    # Limita colunas para caber no gráfico
    top_cols = pivot.max(axis=0).sort_values(ascending=False).head(10).index
    pivot_show = pivot[top_cols]

    fig, ax = plt.subplots(figsize=(14, max(5, len(pivot_show) * 0.55)))
    sns.heatmap(
        pivot_show, annot=True, fmt='.1f', cmap='YlOrRd',
        linewidths=0.4, ax=ax, cbar_kws={'label': 'Lift vs global'}
    )
    ax.set_title('Lift NMF — Projetos × Top 10 Tópicos', fontsize=12, fontweight='bold')
    ax.set_xlabel('Tópico NMF')
    ax.set_ylabel('Projeto')
    plt.xticks(rotation=35, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig(RELATORIO_DIR / 'fig_04_lift_heatmap_projetos.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Insights NMF por projeto não disponíveis — pulando heatmap.')

---
## 7. Evidências Citáveis — Top Trechos por Tópico NMF

Para cada tópico NMF, os 3 trechos com maior score de ativação.
Cada trecho inclui `doc_id` e `chunk_id` para rastreabilidade auditável.

In [9]:
# ── Top trechos por tópico NMF ────────────────────────────────────────────
print('Top-3 trechos representativos por tópico NMF:\n')

for topic in evidencias_json[:5]:  # primeiros 5 para exibição
    print(f"{'─'*60}")
    print(f"[{topic['topic_id']:02d}] {topic['label']}")
    print(f"Termos: {topic['top_terms'][:80]}")
    for ev in topic['evidencias']:
        print(f"  #{ev['rank']} [score={ev['nmf_score']:.3f}] doc={ev['doc_id'][-12:]}")
        print(f"     {ev['trecho'][:200]}...")
    print()

print(f'... e mais {len(evidencias_json) - 5} tópicos — ver outputs/xai/evidencias_por_topico.json')

# Exibe tabela resumo
df_ev_resumo = pd.DataFrame([
    {'topic_id': t['topic_id'], 'label': t['label'],
     'n_evidencias': len(t['evidencias']),
     'max_score': max(e['nmf_score'] for e in t['evidencias'])}
    for t in evidencias_json
])
display(df_ev_resumo)
df_ev_resumo.to_csv(RELATORIO_DIR / 'evidencias_resumo.csv', index=False)

Top-3 trechos representativos por tópico NMF:

────────────────────────────────────────────────────────────
[00] boticario / natura / perfume / vende
Termos: boticario; natura; perfume; vende; marca; marcas; vender; perfumaria; perfumes; 
  #1 [score=0.812] doc=18edb6d745e0
     Se a comissão acabar se igualando, ela vai vender boticário. Bom, é... E tem uma demanda maior com o Boticário? Não, ela tem uma demanda maior com a Natura. São os clientes fiéis dela, a Natura, de an...
  #2 [score=0.681] doc=d290f301eadb
     Olha, são duas excelentes marcas eu acho que ainda existe sabu com a perfumaria da Natura, tá? Sim, diminuindo, né? Mas ainda em relação ao Boticário, perfumes sempre eu vendo mais do Boticário que da...
  #3 [score=0.668] doc=eba5c3deb241
     Pensando mais em produto, além disso que os clientes falam que o Boticário tem melhor duração, ela traz alguma percepção dela também em relação à diferença de produto? Sim, ela pessoalmente gosta de p...

─────────────────────────

,topic_id,label,n_evidencias,max_score
0,0,boticario / natura / perfume / vende,3,0.8122
1,1,propaganda / impressao / propagandas / mulheres,3,0.9506
2,2,idade / trabalho / casa / filhos,3,0.9883
3,3,corpo / banho / uso / hidratante,3,0.9983
4,4,revista / site / compra / comprar,3,0.9877
5,5,presente / dar / presentear / gosta,3,0.9420
6,6,aproveite / sua camera / conversa / local,3,1.0000
7,7,refil / embalagem / parece / formato,3,1.0000
8,8,transmite / atencao / parece / embalagem,3,0.9542
9,9,negra / pele negra / pele / acha,3,0.9990


In [10]:
# ── Taxonomia NB — tema predito por cluster ───────────────────────────────
print('Taxonomia automática (MultinomialNB) por cluster KMeans:\n')

cols_tax = [c for c in ['cluster_label', 'tema_predito', 'confianca', 'tema_2o', 'label_tfidf'] if c in df_taxonomy.columns]
df_tax_show = df_taxonomy[cols_tax].sort_values('confianca', ascending=False)
display(df_tax_show)

n_alta = (df_taxonomy['confianca'] >= 0.5).sum()
print(f'\nClusters com confiança ≥ 0.5: {n_alta}/{len(df_taxonomy)}')
print(f'Confiança média: {df_taxonomy["confianca"].mean():.3f}')
print(f'Temas únicos atribuídos: {df_taxonomy["tema_predito"].nunique()}')

Taxonomia automática (MultinomialNB) por cluster KMeans:



,cluster_label,tema_predito,confianca,tema_2o,label_tfidf
0,km_000,maternidade,1.000,fragancia_aroma,presente / dar / gosta / comprar
7,km_007,cuidados_pele,1.000,experiencia_sensorial,pele / corpo / uso / gosto
16,km_016,cuidados_pele,1.000,experiencia_sensorial,pele negra / negra / pele / linha
3,km_003,embalagem,1.000,design_estetica,refil / embalagem / parece / formato
13,km_013,compra_distribuicao,1.000,preco_custo,revista / compra / site / comprar
12,km_012,maternidade,1.000,sentimento_positivo,tecnologia / bio / bioeficiente / produto
10,km_010,design_estetica,1.000,embalagem,transmite / tecnologia / embalagem / parece
9,km_009,sustentabilidade,0.999,embalagem,produto / marca / produtos / questao
1,km_001,experiencia_sensorial,0.998,design_estetica,pele / propagandas / mulheres / transmite
2,km_002,compra_distribuicao,0.996,marca_branding,loja / boticario / presente / natura



Clusters com confiança ≥ 0.5: 18/18
Confiança média: 0.918
Temas únicos atribuídos: 9


---
## 8. Análise de Sentimento — Síntese

Distribuição de sentimento lexical por projeto e por cluster.
Abordagem principal: léxica com tratamento de negação (sem Deep Learning).

In [11]:
# ── Distribuição geral ────────────────────────────────────────────────────
dist_geral = df_sent_lexico['sentimento_lexico'].value_counts(normalize=True).mul(100).round(1)
print('Distribuição geral de sentimento (4.976 chunks):')
for label, pct in dist_geral.items():
    bar = '█' * int(pct / 2)
    print(f'  {label}  {bar} {pct:.1f}%')

# ── Ranking de projetos por sentimento negativo ───────────────────────────
sent_idx2 = df_sent_proj.copy()
if 'projeto' not in sent_idx2.columns:
    sent_idx2 = sent_idx2.rename(columns={sent_idx2.columns[0]: 'projeto'})
for col in ['POS', 'NEG', 'NEU']:
    if col not in sent_idx2.columns:
        sent_idx2[col] = 0
sent_idx2 = sent_idx2.sort_values('NEG', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras empilhadas
bottom = np.zeros(len(sent_idx2))
cores  = {'POS': '#2ecc71', 'NEU': '#95a5a6', 'NEG': '#e74c3c'}
for col in ['POS', 'NEU', 'NEG']:
    vals = sent_idx2[col].values if col in sent_idx2.columns else np.zeros(len(sent_idx2))
    axes[0].barh(sent_idx2['projeto'], vals, left=bottom, color=cores[col], label=col)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v >= 10:
            axes[0].text(b + v/2, i, f'{v:.0f}%', ha='center', va='center',
                         fontsize=8, color='white', fontweight='bold')
    bottom += vals
axes[0].set_xlabel('% de chunks')
axes[0].set_title('Sentimento por Projeto (% empilhado)', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].invert_yaxis()

# Score médio
_score_proj = df_sent_lexico.groupby('projeto')['score_lexico'].mean().sort_values()
cores_bar   = ['#e74c3c' if v < 0 else '#2ecc71' for v in _score_proj]
axes[1].barh(_score_proj.index, _score_proj.values, color=cores_bar)
axes[1].axvline(0, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Score léxico médio')
axes[1].set_title('Score Léxico Médio por Projeto', fontweight='bold')
for i, (proj, val) in enumerate(_score_proj.items()):
    offset = 0.02 if val >= 0 else -0.02
    ha = 'left' if val >= 0 else 'right'
    axes[1].text(val + offset, i, f'{val:+.2f}', va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.savefig(RELATORIO_DIR / 'fig_05_sentimento_projetos.png', dpi=150, bbox_inches='tight')
plt.show()

Distribuição geral de sentimento (4.976 chunks):
  POS  ████████████████████████████████████████ 81.4%
  NEU  █████ 11.9%
  NEG  ███ 6.7%


In [12]:
# ── Exemplos: trechos mais positivos e mais negativos do corpus ───────────
print('Trechos com maior score positivo (léxico):')
top_pos = df_sent_lexico.nlargest(5, 'score_lexico')[['chunk_id', 'projeto', 'score_lexico', 'sentimento_lexico']]
display(top_pos)

print('\nTrechos com maior score negativo (léxico):')
top_neg = df_sent_lexico.nsmallest(5, 'score_lexico')[['chunk_id', 'projeto', 'score_lexico', 'sentimento_lexico']]
display(top_neg)

# Projeto mais positivo e mais negativo
mais_pos = _score_proj.idxmax()
mais_neg = _score_proj.idxmin()
print(f'\nProjeto mais positivo : {mais_pos} (score médio = {_score_proj[mais_pos]:+.3f})')
print(f'Projeto mais negativo : {mais_neg} (score médio = {_score_proj[mais_neg]:+.3f})')

Trechos com maior score positivo (léxico):


,chunk_id,projeto,score_lexico,sentimento_lexico
3834,doc_9da29b5ddacac7d2_pp_ch_0006,mercato_brasil,25,POS
3798,doc_d9b928d2f9d57ae9_pp_ch_0032,mercato_brasil,24,POS
4848,doc_66b40b2eb6436092_pp_ch_0014,jack_pearson,21,POS
1524,doc_b6fd5d628bb2710e_pp_ch_0020,anima,19,POS
2139,doc_b71f944ee9bec4b0_pp_ch_0045,radiosa,19,POS



Trechos com maior score negativo (léxico):


,chunk_id,projeto,score_lexico,sentimento_lexico
3913,doc_047d9aa2d7261b26_pp_ch_0029,mercato_brasil,-16,NEG
3903,doc_047d9aa2d7261b26_pp_ch_0019,mercato_brasil,-7,NEG
4068,doc_0dc2e6ecde931dc0_pp_ch_0030,mercato_brasil,-7,NEG
4725,doc_eb6a4496f5f5d136_pp_ch_0021,compras_digitais,-7,NEG
4062,doc_0dc2e6ecde931dc0_pp_ch_0024,mercato_brasil,-6,NEG



Projeto mais positivo : jack_pearson (score médio = +5.535)
Projeto mais negativo : bem_maes (score médio = +1.267)


In [13]:
# ── Sentimento por cluster — clusters mais positivos e mais negativos ─────
sent_cl = df_sent_cluster.copy()
if 'cluster_label' not in sent_cl.columns:
    sent_cl = sent_cl.rename(columns={sent_cl.columns[0]: 'cluster_label'})
for col in ['POS', 'NEG', 'NEU']:
    if col not in sent_cl.columns:
        sent_cl[col] = 0

# Junta label TF-IDF
km_lbl_map = dict(zip(df_km_labels['cluster_label'], df_km_labels['auto_label_short']))
sent_cl['label_tfidf'] = sent_cl['cluster_label'].map(km_lbl_map)
sent_cl['label_display'] = sent_cl.apply(
    lambda r: f"{r['cluster_label']} | {str(r.get('label_tfidf',''))[:28]}", axis=1
)

sent_cl_sorted = sent_cl.sort_values('NEG', ascending=False)

fig, ax = plt.subplots(figsize=(8, max(7, len(sent_cl_sorted) * 0.5)))
heat_data = sent_cl_sorted[['POS', 'NEU', 'NEG']].values
im = ax.imshow(heat_data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['POS', 'NEU', 'NEG'], fontsize=11)
ax.set_yticks(range(len(sent_cl_sorted)))
ax.set_yticklabels(sent_cl_sorted['label_display'].values, fontsize=8)
for i in range(len(sent_cl_sorted)):
    for j, col in enumerate(['POS', 'NEU', 'NEG']):
        ax.text(j, i, f"{heat_data[i,j]:.0f}", ha='center', va='center', fontsize=8, color='black')
plt.colorbar(im, ax=ax, label='% chunks')
ax.set_title('Sentimento por Cluster KMeans', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RELATORIO_DIR / 'fig_06_sentimento_por_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Validação do Modelo

Resumo dos experimentos de validação: classificação supervisionada, baseline honesto,
leakage semântico e análise de erros.

In [14]:
# ── Comparação de modelos ─────────────────────────────────────────────────
print('Classificação Supervisionada — Predição de Projeto (alvo: projeto)\n')

if not df_baseline.empty:
    df_val_show = df_baseline.copy()
    if 'f1_macro' not in df_val_show.columns:
        # pode vir como f1_macro ou f1_macro_media
        f1_col = next((c for c in df_val_show.columns if 'f1' in c.lower()), None)
        acc_col= next((c for c in df_val_show.columns if 'acc' in c.lower()), None)
        if f1_col:
            df_val_show = df_val_show.rename(columns={f1_col: 'f1_macro'})
        if acc_col:
            df_val_show = df_val_show.rename(columns={acc_col: 'accuracy'})
    cols_v = [c for c in ['modelo', 'accuracy', 'f1_macro', 'observacao'] if c in df_val_show.columns]
    display(df_val_show[cols_v].sort_values('f1_macro', ascending=False))
else:
    # Fallback para o CSV do NB 04 sem seção 10
    df_class_show = df_class.copy()
    f1_col = next((c for c in df_class_show.columns if 'f1' in c.lower()), None)
    if f1_col:
        df_class_show = df_class_show.sort_values(f1_col, ascending=False)
    display(df_class_show)

# Notas de interpretação
print()
print('Interpretação:')
print('  • DummyClassifier (baseline trivial): prevê sempre a classe mais frequente')
print('  • F1-macro mede performance em classes desbalanceadas')
print('  • LinearSVC e LogReg superam o baseline com margem grande → modelo aprende padrões reais')

Classificação Supervisionada — Predição de Projeto (alvo: projeto)



,modelo,accuracy,f1_macro,observacao
0,LinearSVC,0.9518,0.9557,SVM linear — melhor modelo (ver seção 6)
1,LogReg,0.9271,0.9333,Regressão Logística — linear com regularização
2,RandomForest,0.8531,0.8548,"Random Forest — não-linear, detecta interações entre termos"
3,MultinomialNB,0.7195,0.6240,Naive Bayes — baseline probabilístico simples
4,DummyClassifier,0.1980,0.0275,Baseline trivial — sempre prevê a classe mais frequente



Interpretação:
  • DummyClassifier (baseline trivial): prevê sempre a classe mais frequente
  • F1-macro mede performance em classes desbalanceadas
  • LinearSVC e LogReg superam o baseline com margem grande → modelo aprende padrões reais


In [15]:
# ── Teste de Leakage Semântico ────────────────────────────────────────────
if not df_leakage.empty:
    print('Diagnóstico de Leakage Semântico (LinearSVC):\n')
    cols_lk = [c for c in [
        'modelo', 'f1_macro_com_suspeitos', 'f1_macro_sem_suspeitos', 'delta',
        'termos_testados', 'pct_chunks_afetados', 'conclusao'
    ] if c in df_leakage.columns]
    display(df_leakage[cols_lk])
else:
    print('Arquivo de diagnóstico de leakage não encontrado — verifique NB04 seção 10.')

Diagnóstico de Leakage Semântico (LinearSVC):



,modelo,f1_macro_com_suspeitos,f1_macro_sem_suspeitos,delta,termos_testados,pct_chunks_afetados,conclusao
0,LinearSVC,0.9557,0.9564,0.0006,"natura, avon, boticario, boticário, radiosa, kyra",46.7,Sem queda relevante: F1 alto é sustentado pelo conteúdo semântico real.


In [16]:
# ── Análise de Erros ──────────────────────────────────────────────────────
if not df_erros.empty:
    n_erros = len(df_erros)
    print(f'Amostra de {n_erros} exemplos de erro do melhor modelo:\n')
    cols_er = [c for c in ['projeto_real', 'projeto_previsto', 'possivel_causa', 'trecho'] if c in df_erros.columns]
    display(df_erros[cols_er].head(10))

    # Quais pares erram mais?
    if 'projeto_real' in df_erros.columns and 'projeto_previsto' in df_erros.columns:
        top_pares = (
            df_erros.groupby(['projeto_real', 'projeto_previsto'])
            .size()
            .reset_index(name='qtd_erros')
            .sort_values('qtd_erros', ascending=False)
            .head(10)
        )
        print('\nPares com mais erros:')
        display(top_pares)
else:
    print('Arquivo de exemplos de erros não encontrado — verifique NB04 seção 10.')

Amostra de 20 exemplos de erro do melhor modelo:



,projeto_real,projeto_previsto,possivel_causa,trecho
0,biome,radiosa,Projetos com vocabulário temático similar,"Como eu falei, pra mim foi diferente, porque eu tô na... A minha vibe atual é você comprou, usa até o fim. Eu tô ten..."
1,biome,compras_digitais,Termos genéricos ou sobreposição de contexto,"Normalmente sempre comprou lá no Amazon MercadoLibre, no geral lá no Amazon, mas realmente assim, lojinhas pequenas ..."
2,anima,radiosa,Projetos com vocabulário temático similar,"Porque, meu, a gente vem entender tudo que vocês falam, né, e nessa história, tipo, vocês falam todo mundo ao mesmo ..."
3,gaia_ii,rosacea,Projetos com vocabulário temático similar,"Olha, eu... A televisão... Fica ligada 24 horas quando eu tô em casa. Então, assim, eu não sou muito fã, mas como eu..."
4,radiosa,natura_3cs,Termos genéricos ou sobreposição de contexto,"Fala pra gente. Sim. Então, meu público primeiro é a família, aí tem a vizinhança, a escola das crianças, e eu tenho..."
5,natura_3cs,mercato_brasil,Termos genéricos ou sobreposição de contexto,"O que elas pedem? Neodora de shampoo? É, da Eudora que eu tô vendendo mais é, assim, eu vendendo não, né? Aquelas pe..."
6,compras_digitais,natura_3cs,Termos genéricos ou sobreposição de contexto,"Além de Natura, mais alguma marca que você usa? Eu assino uma box. E aí eles mandam produtos às vezes. É uma box mag..."
7,3cs_perfumes,natura_3cs,Projetos com vocabulário temático similar,"A Avon, para mim... Ela seria... Como se fosse, meu Deus do céu, alguém que começou toda a minha trajetória, sabe? T..."
8,biome,radiosa,Projetos com vocabulário temático similar,"Quem quer começar? Fica super à vontade. Posso começar? Vai lá, Fê. Meu nome é Fernanda. eu sou [NOME] e trabalho co..."
9,natura_3cs,mercato_brasil,Termos genéricos ou sobreposição de contexto,"Olha, até que tem. Até aqui sei. Que bom. Assim, como eu disse, eu tô natura há bom tempo. E essa função, ela me... ..."



Pares com mais erros:


,projeto_real,projeto_previsto,qtd_erros
4,biome,radiosa,3
10,natura_3cs,3cs_perfumes,3
0,3cs_perfumes,natura_3cs,2
8,gaia_ii,rosacea,2
11,natura_3cs,mercato_brasil,2
1,anima,radiosa,1
2,bem_maes,compras_digitais,1
3,biome,compras_digitais,1
5,compras_digitais,biome,1
6,compras_digitais,natura_3cs,1


---
## 10. Export Final

Consolida todos os resultados em arquivos de entrega:
- `relatorio_final.json` — estrutura completa para uso programático (app.py, API)
- `relatorio_final.md` — relatório legível por humanos
- `tabela_insights_final.csv` — tabela de insights com lift para o cliente
- `tabela_evidencias_final.csv` — evidências citáveis por tópico

In [17]:
# ── Monta estrutura JSON final ───────────────────────────────────────────

# Sentimento geral
dist_geral_d = df_sent_lexico['sentimento_lexico'].value_counts(normalize=True).mul(100).round(1).to_dict()

# Resumo por projeto
proj_payload = {}
for proj in df_base['projeto'].dropna().unique():
    sub_base = df_base[df_base['projeto'] == proj]
    sub_sent = df_sent_lexico[df_sent_lexico['projeto'] == proj] if 'projeto' in df_sent_lexico.columns else pd.DataFrame()

    # Tema NMF dominante
    tema_info = tema_dom_proj.loc[proj] if proj in tema_dom_proj.index else {}

    # Sentimento
    if not sub_sent.empty:
        vc_s = sub_sent['sentimento_lexico'].value_counts()
        total_s = len(sub_sent)
        sent_d = {
            'pct_pos': round(vc_s.get('POS', 0) / total_s * 100, 1),
            'pct_neg': round(vc_s.get('NEG', 0) / total_s * 100, 1),
            'pct_neu': round(vc_s.get('NEU', 0) / total_s * 100, 1),
            'score_medio': round(sub_sent['score_lexico'].mean(), 2),
            'dominante': vc_s.idxmax() if len(vc_s) > 0 else 'NEU',
        }
    else:
        sent_d = {}

    # Top termos Chi²
    top_termos = []
    if _chi2_path.exists():
        df_chi2_local = pd.read_csv(_chi2_path)
        top_termos = df_chi2_local[df_chi2_local['classe'] == proj].nsmallest(10, 'rank')['termo'].tolist()

    # Top 3 insights lift
    top_insights = []
    if 'segment_value' in df_ins_km.columns and 'lift_vs_global' in df_ins_km.columns:
        sub_ins = df_ins_km[
            (df_ins_km['segment_value'] == proj) &
            (df_ins_km['segment_col'] == 'projeto')
        ].nlargest(3, 'lift_vs_global')
        for _, r in sub_ins.iterrows():
            top_insights.append({
                'tema': r.get('auto_label', ''),
                'lift': round(float(r.get('lift_vs_global', 0)), 2),
                'share': round(float(r.get('share_in_segment', 0)) * 100, 1),
            })

    proj_payload[proj] = {
        'n_chunks':     int(len(sub_base)),
        'n_docs':       int(sub_base['doc_id'].nunique()),
        'tema_dominante': {
            'id':    str(tema_info.get('tema_nmf_id', '')),
            'label': str(tema_info.get('tema_nmf_label', '')),
        },
        'sentimento':   sent_d,
        'top_termos':   top_termos[:10],
        'top_insights': top_insights,
    }

# Validação
val_payload = {
    'melhor_modelo': df_class.iloc[0].to_dict() if len(df_class) > 0 else {},
    'topicos_robustos_n': int(len(df_jaccard)),
    'topicos_robustos': df_jaccard[['lda_label', 'nmf_label', 'jaccard']].to_dict('records') if len(df_jaccard) > 0 else [],
    'coerencia_cv_melhor_k': int(df_coh.loc[df_coh['coerencia_cv'].idxmax(), 'k']) if len(df_coh) > 0 else None,
}

# Payload final
relatorio_final = {
    'meta': {
        'gerado_em': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'run_id':    CLUSTER_RUN.name,
        'pipeline':  'Kyra Pesquisa — NLP Pipeline 2026',
        'equipe':    'Maria Beatriz Ribeiro, Juliane Oliveira, Emanuel Gandra',
    },
    'corpus': kpis,
    'sentimento_geral': dist_geral_d,
    'projetos': proj_payload,
    'validacao': val_payload,
    'evidencias': evidencias_json,
}

json_path = RELATORIO_DIR / 'relatorio_final.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(relatorio_final, f, ensure_ascii=False, indent=2)

print(f'relatorio_final.json salvo → {json_path}')
print(f'  Projetos cobertos : {len(proj_payload)}')
print(f'  Tópicos NMF       : {len(evidencias_json)}')

relatorio_final.json salvo → /Users/I759133/Library/CloudStorage/OneDrive-SAPSE/Documentos Juliane/IBMEC/SEXTO SEMESTRE/PROJETO/kyrapesquisa/Kyra Atual/outputs/relatorio_final/relatorio_final.json
  Projetos cobertos : 12
  Tópicos NMF       : 18


In [18]:
# ── Gera relatorio_final.md ───────────────────────────────────────────────
gerado_em = relatorio_final['meta']['gerado_em']

linhas_md = [
    '# Kyra Pesquisa — Relatório Final',
    '',
    f'**Gerado em:** {gerado_em}  ',
    f'**Run ID:** `{CLUSTER_RUN.name}`  ',
    f'**Equipe:** Maria Beatriz Ribeiro · Juliane Oliveira · Emanuel Gandra',
    '',
    '---',
    '',
    '## 1. Visão Geral do Corpus',
    '',
    f'| Métrica | Valor |',
    f'|---|---|',
    f'| Chunks analisados | {kpis["chunks_analisados"]:,} |',
    f'| Documentos | {kpis["documentos"]} |',
    f'| Projetos | {kpis["projetos"]} |',
    f'| Clusters KMeans | {kpis["clusters_kmeans"]} |',
    f'| Tópicos NMF | {kpis["topicos_nmf"]} |',
    f'| Tópicos LDA | {kpis["topicos_lda"]} |',
    f'| Sentimento geral (POS) | {kpis["pct_pos_geral"]:.1f}% |',
    f'| Sentimento geral (NEU) | {kpis["pct_neu_geral"]:.1f}% |',
    f'| Sentimento geral (NEG) | {kpis["pct_neg_geral"]:.1f}% |',
    '',
    '---',
    '',
    '## 2. Resumo por Projeto',
    '',
    '| Projeto | Chunks | Tema Dominante (NMF) | % POS | % NEG | Score Léxico |',
    '|---|---:|---|---:|---:|---:|',
]

for proj, info in sorted(proj_payload.items(), key=lambda x: -x[1]['n_chunks']):
    tema_lbl = info['tema_dominante'].get('label', '—')[:35]
    s = info.get('sentimento', {})
    linhas_md.append(
        f"| {proj} | {info['n_chunks']:,} | {tema_lbl} | "
        f"{s.get('pct_pos','—')} | {s.get('pct_neg','—')} | {s.get('score_medio','—')} |"
    )

linhas_md += [
    '',
    '---',
    '',
    '## 3. Tópicos Robustos (LDA × NMF, Jaccard ≥ 0.25)',
    '',
    f'{len(df_jaccard)}/{df_lda.shape[0]} tópicos LDA confirmados por correspondente NMF.',
    '',
    '| Tópico LDA | Tópico NMF | Jaccard | Termos Comuns |',
    '|---|---|---:|---|',
]
for _, r in df_jaccard.sort_values('jaccard', ascending=False).iterrows():
    linhas_md.append(
        f"| {r.get('lda_label','')[:35]} | {r.get('nmf_label','')[:35]} | "
        f"{r.get('jaccard', 0):.2f} | {r.get('termos_comuns','')[:40]} |"
    )

linhas_md += [
    '',
    '---',
    '',
    '## 4. Validação do Modelo',
    '',
    '**Classificação supervisionada (alvo: projeto):**',
    '',
]

_val = relatorio_final['validacao']
if _val.get('melhor_modelo'):
    m = _val['melhor_modelo']
    f1_key = next((k for k in m if 'f1' in k.lower()), None)
    if f1_key:
        linhas_md.append(f"Melhor modelo: **{m.get('modelo','—')}** | F1-macro = **{m.get(f1_key,'—')}**")

if not df_baseline.empty:
    f1_col = next((c for c in df_baseline.columns if 'f1' in c.lower()), None)
    acc_col= next((c for c in df_baseline.columns if 'acc' in c.lower()), None)
    linhas_md += ['', '| Modelo | Accuracy | F1-macro |', '|---|---:|---:|']
    for _, r in df_baseline.sort_values(f1_col or df_baseline.columns[1], ascending=False).iterrows():
        linhas_md.append(
            f"| {r.get('modelo','—')} | {r.get(acc_col or '','—')} | {r.get(f1_col or '','—')} |"
        )

linhas_md += [
    '',
    '---',
    '',
    '## 5. Evidências Citáveis (top-3 por tópico NMF)',
    '',
]
for t in evidencias_json[:6]:  # primeiros 6 tópicos
    linhas_md.append(f"### Tópico {t['topic_id']:02d} — {t['label']}")
    linhas_md.append(f"**Termos:** {t['top_terms'][:80]}")
    linhas_md.append('')
    for ev in t['evidencias']:
        linhas_md.append(f"> #{ev['rank']} `{ev['chunk_id']}` (score={ev['nmf_score']:.3f})")
        linhas_md.append(f"> {ev['trecho'][:300]}")
        linhas_md.append('')

md_path = RELATORIO_DIR / 'relatorio_final.md'
md_path.write_text('\n'.join(linhas_md), encoding='utf-8')
print(f'relatorio_final.md salvo → {md_path}  ({len(linhas_md)} linhas)')

relatorio_final.md salvo → /Users/I759133/Library/CloudStorage/OneDrive-SAPSE/Documentos Juliane/IBMEC/SEXTO SEMESTRE/PROJETO/kyrapesquisa/Kyra Atual/outputs/relatorio_final/relatorio_final.md  (152 linhas)


In [19]:
# ── Tabelas CSV de entrega ────────────────────────────────────────────────

# 1. Insights consolidados (clusters + NMF)
df_ins_km.to_csv(RELATORIO_DIR / 'tabela_insights_clusters.csv', index=False)
df_ins_nmf.to_csv(RELATORIO_DIR / 'tabela_insights_nmf.csv', index=False)

# 2. Evidências citáveis por tópico
df_evidencias.to_csv(RELATORIO_DIR / 'tabela_evidencias_final.csv', index=False)

# 3. Resumo sentimento por projeto
df_sent_proj.to_csv(RELATORIO_DIR / 'tabela_sentimento_por_projeto.csv', index=False)

# 4. Exemplos de sentimento (POS/NEG por projeto)
df_exemplos_sent.to_csv(RELATORIO_DIR / 'tabela_exemplos_sentimento.csv', index=False)

# 5. Tópicos robustos
df_jaccard.to_csv(RELATORIO_DIR / 'tabela_topicos_robustos.csv', index=False)

# 6. Resumo validação
if not df_baseline.empty:
    df_baseline.to_csv(RELATORIO_DIR / 'tabela_validacao_modelos.csv', index=False)
else:
    df_class.to_csv(RELATORIO_DIR / 'tabela_validacao_modelos.csv', index=False)

# 7. Corpus por projeto
proj_dist.to_csv(RELATORIO_DIR / 'tabela_corpus_por_projeto.csv', index=False)

print('Arquivos de entrega salvos em outputs/relatorio_final/:')
for f in sorted(RELATORIO_DIR.glob('*')):
    size = f.stat().st_size
    unit = 'KB' if size < 1_000_000 else 'MB'
    size_str = f'{size/1024:.1f} {unit}' if size < 1_000_000 else f'{size/1_048_576:.1f} {unit}'
    print(f'  {f.name:<45} {size_str}')

Arquivos de entrega salvos em outputs/relatorio_final/:
  corpus_por_projeto.csv                        0.3 KB
  evidencias_resumo.csv                         0.9 KB
  fig_01_chunks_por_projeto.png                 71.0 KB
  fig_02_sentimento_por_projeto.png             90.0 KB
  fig_03_jaccard_robustos.png                   87.4 KB
  fig_04_lift_heatmap_projetos.png              72.2 KB
  fig_05_sentimento_projetos.png                115.2 KB
  fig_06_sentimento_por_cluster.png             119.0 KB
  insights_top_por_segmento.csv                 22.2 KB
  relatorio_final.json                          61.4 KB
  relatorio_final.md                            10.7 KB
  resumo_por_projeto.csv                        1.8 KB
  tabela_corpus_por_projeto.csv                 0.3 KB
  tabela_evidencias_final.csv                   45.9 KB
  tabela_exemplos_sentimento.csv                34.3 KB
  tabela_insights_clusters.csv                  168.7 KB
  tabela_insights_nmf.csv                       1

---
## 11. Resumo Final do Pipeline

In [20]:
print('=' * 65)
print('RESUMO FINAL — KYRA PESQUISA NLP PIPELINE 2026')
print('=' * 65)

print(f'''
[Pipeline completo]
  NB 01 — Diagnóstico    : {n_docs} documentos auditados | {n_chunks:,} chunks brutos
  NB 02 — Pré-proc       : {n_chunks:,} chunks limpos em PT sem ruído residual
  NB 03 — Clusterização  : KMeans k={n_clusters} | NMF {n_nmf} tópicos | LDA {n_lda} tópicos
  NB 04 — Validação      : LinearSVC f1_macro=0.956 | {len(df_jaccard)}/{n_lda} pares robustos Jaccard
  NB 05 — XAI            : {len(evidencias_json)} tópicos com evidência citável | PII: base limpa
  NB 06 — Sentimento     : Léxico {dist_geral_d.get('POS',0):.1f}% POS / {dist_geral_d.get('NEU',0):.1f}% NEU / {dist_geral_d.get('NEG',0):.1f}% NEG
  NB 07 — Relatório      : {len(proj_payload)} projetos consolidados → outputs/relatorio_final/
''')

print('[Outputs de entrega]')
for f in sorted(RELATORIO_DIR.glob('*')):
    print(f'  └─ {f.name}')

print(f'''
[Próximos passos recomendados]
  1. Implementar src/pipeline.py — conectar modelos ao app.py (hoje usa mock_pipeline)
  2. Validar rótulos NMF manualmente (ver outputs/xai/top_trechos_nmf.csv)
  3. Revisar clusters com baixa confiança NB (outputs/xai/clusters_taxonomy_nb.csv)
  4. Preparar slides de apresentação usando as figuras em outputs/relatorio_final/
''')

RESUMO FINAL — KYRA PESQUISA NLP PIPELINE 2026

[Pipeline completo]
  NB 01 — Diagnóstico    : 131 documentos auditados | 4,976 chunks brutos
  NB 02 — Pré-proc       : 4,976 chunks limpos em PT sem ruído residual
  NB 03 — Clusterização  : KMeans k=18 | NMF 18 tópicos | LDA 10 tópicos
  NB 04 — Validação      : LinearSVC f1_macro=0.956 | 10/10 pares robustos Jaccard
  NB 05 — XAI            : 18 tópicos com evidência citável | PII: base limpa
  NB 06 — Sentimento     : Léxico 81.4% POS / 11.9% NEU / 6.7% NEG
  NB 07 — Relatório      : 12 projetos consolidados → outputs/relatorio_final/

[Outputs de entrega]
  └─ corpus_por_projeto.csv
  └─ evidencias_resumo.csv
  └─ fig_01_chunks_por_projeto.png
  └─ fig_02_sentimento_por_projeto.png
  └─ fig_03_jaccard_robustos.png
  └─ fig_04_lift_heatmap_projetos.png
  └─ fig_05_sentimento_projetos.png
  └─ fig_06_sentimento_por_cluster.png
  └─ insights_top_por_segmento.csv
  └─ relatorio_final.json
  └─ relatorio_final.md
  └─ resumo_por_projeto.